# Ocean Model-comparison Bootstrap Experiment

Runs the eight fixed candidate models across randomized train/test split seeds. For each split and candidate, the notebook fits 20 EM starts, keeps the start with the lowest training BIC, and records train and held-out metrics plus split and parameter audit metadata.

In [1]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import json
import numpy as np
import pandas as pd

import experiment.ocean.helpers as mod

In [2]:
PROJECT_ROOT = mod.find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "ndbc" / "ocean_data.csv"
RESULTS_DIR = PROJECT_ROOT / "results" / "ocean"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_SEEDS = list(range(100))
TRAIN_FRACTION = 0.8
N_EM_STARTS = 1
EM_SEED_BASE = 1729

INIT = "k-means"
INIT_LAYER1 = "k-means"
INIT_LAYER2 = "k-means"
TOL = 1e-4
MAX_ITER = 1000
C_STEP_BOOL = False
SHOW_PROGRESS = True

In [3]:
CANDIDATES = [
    {
        "candidate_id": "ind_cyl_k3",
        "display_name": "Independent cylindrical mixture, K=3",
        "model_type": "cylindrical_mixture",
        "family": "independent",
        "k": 3,
        "l": None,
    },
    {
        "candidate_id": "ind_cyl_k6",
        "display_name": "Independent cylindrical mixture, K=6",
        "model_type": "cylindrical_mixture",
        "family": "independent",
        "k": 6,
        "l": None,
    },
    {
        "candidate_id": "full_cyl_k3",
        "display_name": "Full cylindrical mixture, K=3",
        "model_type": "cylindrical_mixture",
        "family": "full",
        "k": 3,
        "l": None,
    },
    {
        "candidate_id": "full_cyl_k6",
        "display_name": "Full cylindrical mixture, K=6",
        "model_type": "cylindrical_mixture",
        "family": "full",
        "k": 6,
        "l": None,
    },
    {
        "candidate_id": "gauss_vmf_tl_3x2",
        "display_name": "Gaussian/vMF two-layer, (K,L)=(3,2)",
        "model_type": "two_layer_mixture",
        "family": "gaussian_vmf",
        "k": 3,
        "l": 2,
    },
    {
        "candidate_id": "gauss_vmf_tl_6x4",
        "display_name": "Gaussian/vMF two-layer, (K,L)=(6,4)",
        "model_type": "two_layer_mixture",
        "family": "gaussian_vmf",
        "k": 6,
        "l": 4,
    },
    {
        "candidate_id": "vmf_gauss_tl_3x2",
        "display_name": "vMF/Gaussian two-layer, (K,L)=(3,2)",
        "model_type": "two_layer_mixture",
        "family": "vmf_gaussian",
        "k": 3,
        "l": 2,
    },
    {
        "candidate_id": "vmf_gauss_tl_6x4",
        "display_name": "vMF/Gaussian two-layer, (K,L)=(6,4)",
        "model_type": "two_layer_mixture",
        "family": "vmf_gaussian",
        "k": 6,
        "l": 4,
    },
]

pd.DataFrame(CANDIDATES)

,candidate_id,display_name,model_type,family,k,l
0,ind_cyl_k3,"Independent cylindrical mixture, K=3",cylindrical_mixture,independent,3,NaN
1,ind_cyl_k6,"Independent cylindrical mixture, K=6",cylindrical_mixture,independent,6,NaN
2,full_cyl_k3,"Full cylindrical mixture, K=3",cylindrical_mixture,full,3,NaN
3,full_cyl_k6,"Full cylindrical mixture, K=6",cylindrical_mixture,full,6,NaN
4,gauss_vmf_tl_3x2,"Gaussian/vMF two-layer, (K,L)=(3,2)",two_layer_mixture,gaussian_vmf,3,2.0
5,gauss_vmf_tl_6x4,"Gaussian/vMF two-layer, (K,L)=(6,4)",two_layer_mixture,gaussian_vmf,6,4.0
6,vmf_gauss_tl_3x2,"vMF/Gaussian two-layer, (K,L)=(3,2)",two_layer_mixture,vmf_gaussian,3,2.0
7,vmf_gauss_tl_6x4,"vMF/Gaussian two-layer, (K,L)=(6,4)",two_layer_mixture,vmf_gaussian,6,4.0


In [4]:
def make_em_start_seeds(split_seed: int, candidate_index: int) -> list[int]:
    rng_seed = int(
        (EM_SEED_BASE + 1009 * int(split_seed) + 101 * int(candidate_index))
        % np.iinfo(np.uint32).max
    )
    rng = np.random.RandomState(rng_seed)
    return rng.randint(np.iinfo(np.int32).max, size=N_EM_STARTS).astype(int).tolist()


def fit_candidate(candidate: dict, prepared: dict, split_seed: int, candidate_index: int) -> dict:
    start_seeds = make_em_start_seeds(split_seed, candidate_index)
    common = {
        "n_starts": N_EM_STARTS,
        "start_seeds": start_seeds,
        "tol": TOL,
        "max_iter": MAX_ITER,
        "c_step_bool": C_STEP_BOOL,
        "show_progress": SHOW_PROGRESS,
        "fail_fast": False,
    }

    if candidate["model_type"] == "cylindrical_mixture":
        return mod.fit_best_cylindrical_mixture(
            prepared["x_train"],
            prepared["x_test"],
            d_gauss=prepared["d_gauss"],
            d_vmf=prepared["d_vmf"],
            n_components=candidate["k"],
            family=candidate["family"],
            init=INIT,
            **common,
        )

    if candidate["family"] == "gaussian_vmf":
        return mod.fit_best_two_layer_mixture(
            prepared["x_train_linear"],
            prepared["x_train_direction"],
            prepared["x_test_linear"],
            prepared["x_test_direction"],
            family="gaussian_vmf",
            n_layer1_components=candidate["k"],
            n_layer2_components=candidate["l"],
            init_layer1=INIT_LAYER1,
            init_layer2=INIT_LAYER2,
            **common,
        )

    if candidate["family"] == "vmf_gaussian":
        return mod.fit_best_two_layer_mixture(
            prepared["x_train_direction"],
            prepared["x_train_linear"],
            prepared["x_test_direction"],
            prepared["x_test_linear"],
            family="vmf_gaussian",
            n_layer1_components=candidate["k"],
            n_layer2_components=candidate["l"],
            init_layer1=INIT_LAYER1,
            init_layer2=INIT_LAYER2,
            **common,
        )

    raise ValueError(f"Unknown candidate: {candidate}")

In [5]:
BEST_TABLE_EXCLUDED_COLUMNS = {
    "parameter_summary",
    "start_seeds",
    "k",
    "l",
    "n_components",
    "n_layer1_components",
    "n_layer2_components",
}


best_table_records = []
selected_records = []
start_records = []
split_records = []
fitted_models = {}

for split_seed in SPLIT_SEEDS:
    prepared = mod.prepare_data(
        csv_path=DATA_PATH,
        train_fraction=TRAIN_FRACTION,
        shuffle=True,
        random_state=split_seed,
    )
    split_records.append({
        "split_seed": int(split_seed),
        "train_fraction": float(TRAIN_FRACTION),
        "shuffle": True,
        "n_train": int(prepared["x_train"].shape[0]),
        "n_test": int(prepared["x_test"].shape[0]),
        "d_gauss": int(prepared["d_gauss"]),
        "d_vmf": int(prepared["d_vmf"]),
        "linear_features": list(prepared["linear_features"]),
        "direction_feature": prepared["direction_feature"],
        "train_time_range": [str(value) for value in prepared["train_time_range"]],
        "test_time_range": [str(value) for value in prepared["test_time_range"]],
        "train_row_ids": prepared["train_row_ids"].astype(int).tolist(),
        "test_row_ids": prepared["test_row_ids"].astype(int).tolist(),
    })
    print(f"Split seed {split_seed}: n_train={prepared['x_train'].shape[0]}, n_test={prepared['x_test'].shape[0]}")

    for candidate_index, candidate in enumerate(CANDIDATES):
        print(f"Fitting {candidate['candidate_id']} for split seed {split_seed}")
        result = fit_candidate(candidate, prepared, split_seed, candidate_index)
        fitted_models[(split_seed, candidate["candidate_id"])] = result["best_model"]

        base_meta = {
            "split_seed": int(split_seed),
            "candidate_id": candidate["candidate_id"],
            "display_name": candidate["display_name"],
            "candidate_model_type": candidate["model_type"],
            "candidate_family": candidate["family"],
            "K": int(candidate["k"]),
            "L": None if candidate["l"] is None else int(candidate["l"]),
            "selection_metric": result["selection_metric"],
            "n_em_starts": int(result["n_starts"]),
        }

        best = result["best_result"].copy()
        best_parameter_summary = best.pop("parameter_summary", None)
        selected = {**base_meta, **best}
        selected["start_seeds"] = [int(seed) for seed in result["start_seeds"]]
        selected["parameter_summary"] = best_parameter_summary
        selected_records.append(selected)

        table_record = {key: value for key, value in selected.items() if key not in BEST_TABLE_EXCLUDED_COLUMNS}
        best_table_records.append(table_record)

        for start_result in result["results"]:
            start_record = start_result.copy()
            start_record["parameter_summary"] = start_record.get("parameter_summary")
            start_records.append({**base_meta, **start_record})

best_df = pd.DataFrame(best_table_records)
best_df

Split seed 0: n_train=149792, n_test=37449
Fitting ind_cyl_k3 for split seed 0
Finished EM start 1/1 (100.0%) | independent cylindrical K=3 | start=0 | seed=911214221 | train_BIC=1.18389e+06 | heldout_LL=-147928 | success
Fitting ind_cyl_k6 for split seed 0
Finished EM start 1/1 (100.0%) | independent cylindrical K=6 | start=0 | seed=403708358 | train_BIC=1.14595e+06 | heldout_LL=-143364 | success
Fitting full_cyl_k3 for split seed 0
Finished EM start 1/1 (100.0%) | full cylindrical K=3 | start=0 | seed=40921045 | train_BIC=1.16642e+06 | heldout_LL=-145874 | success
Fitting full_cyl_k6 for split seed 0
Finished EM start 1/1 (100.0%) | full cylindrical K=6 | start=0 | seed=1956696334 | train_BIC=1.14164e+06 | heldout_LL=-142786 | success
Fitting gauss_vmf_tl_3x2 for split seed 0
Finished EM start 1/1 (100.0%) | gaussian_vmf two-layer K=3 L=2 | start=0 | seed=329594251 | train_BIC=1.16955e+06 | heldout_LL=-146141 | success
Fitting gauss_vmf_tl_6x4 for split seed 0
Finished EM start 1/1 (

,split_seed,candidate_id,display_name,candidate_model_type,candidate_family,candidate_k,candidate_l,selection_metric,n_em_starts,model_type,...,heldout_gmpd,n_free_params,n_iter,converged,fit_time,n_train,n_test,l,n_layer1_components,n_layer2_components
0,0,ind_cyl_k3,"Independent cylindrical mixture, K=3",cylindrical_mixture,independent,3,NaN,train_bic,1,cylindrical_mixture,...,0.019252,35,33,True,9.076698,149792,37449,NaN,NaN,NaN
1,0,ind_cyl_k6,"Independent cylindrical mixture, K=6",cylindrical_mixture,independent,6,NaN,train_bic,1,cylindrical_mixture,...,0.021748,71,33,True,16.968939,149792,37449,NaN,NaN,NaN
2,0,full_cyl_k3,"Full cylindrical mixture, K=3",cylindrical_mixture,full,3,NaN,train_bic,1,cylindrical_mixture,...,0.020338,53,27,True,9.088113,149792,37449,NaN,NaN,NaN
3,0,full_cyl_k6,"Full cylindrical mixture, K=6",cylindrical_mixture,full,6,NaN,train_bic,1,cylindrical_mixture,...,0.022086,107,69,True,27.985685,149792,37449,NaN,NaN,NaN
4,0,gauss_vmf_tl_3x2,"Gaussian/vMF two-layer, (K,L)=(3,2)",two_layer_mixture,gaussian_vmf,3,2.0,train_bic,1,two_layer_mixture,...,0.020193,44,46,True,20.307803,149792,37449,2.0,3.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,99,full_cyl_k6,"Full cylindrical mixture, K=6",cylindrical_mixture,full,6,NaN,train_bic,1,cylindrical_mixture,...,0.022059,107,52,True,22.178662,149792,37449,NaN,NaN,NaN
796,99,gauss_vmf_tl_3x2,"Gaussian/vMF two-layer, (K,L)=(3,2)",two_layer_mixture,gaussian_vmf,3,2.0,train_bic,1,two_layer_mixture,...,0.020062,44,65,True,26.725885,149792,37449,2.0,3.0,2.0
797,99,gauss_vmf_tl_6x4,"Gaussian/vMF two-layer, (K,L)=(6,4)",two_layer_mixture,gaussian_vmf,6,4.0,train_bic,1,two_layer_mixture,...,0.022897,125,46,True,47.376309,149792,37449,4.0,6.0,4.0
798,99,vmf_gauss_tl_3x2,"vMF/Gaussian two-layer, (K,L)=(3,2)",two_layer_mixture,vmf_gaussian,3,2.0,train_bic,1,two_layer_mixture,...,0.022073,65,24,True,11.043528,149792,37449,2.0,3.0,2.0


In [6]:
def write_jsonl(path, records) -> None:
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, default=str, allow_nan=True) + "\n")


best_csv_path = RESULTS_DIR / "ocean_model_comparison_bootstrap_best.csv"
selected_jsonl_path = RESULTS_DIR / "ocean_model_comparison_bootstrap_selected.jsonl"
starts_jsonl_path = RESULTS_DIR / "ocean_model_comparison_bootstrap_all_starts.jsonl"
splits_json_path = RESULTS_DIR / "ocean_model_comparison_bootstrap_splits.json"
splits_csv_path = RESULTS_DIR / "ocean_model_comparison_bootstrap_splits.csv"

best_df = best_df.copy()
if "K" not in best_df.columns:
    k_source = "candidate_k" if "candidate_k" in best_df.columns else "k"
    best_df.insert(best_df.columns.get_loc(k_source), "K", best_df[k_source])
if "L" not in best_df.columns:
    l_source = "candidate_l" if "candidate_l" in best_df.columns else "l"
    best_df.insert(best_df.columns.get_loc(l_source), "L", best_df[l_source])
best_df = best_df.drop(
    columns=[
        column
        for column in (
            "candidate_k",
            "candidate_l",
            "k",
            "l",
            "n_components",
            "n_layer1_components",
            "n_layer2_components",
        )
        if column in best_df.columns
    ]
)

best_df.to_csv(best_csv_path, index=False)
write_jsonl(selected_jsonl_path, selected_records)
write_jsonl(starts_jsonl_path, start_records)

with splits_json_path.open("w", encoding="utf-8") as handle:
    json.dump(split_records, handle, default=str, allow_nan=True)

split_table = pd.DataFrame([
    {key: value for key, value in record.items() if key not in {"train_row_ids", "test_row_ids"}}
    for record in split_records
])
split_table.to_csv(splits_csv_path, index=False)

pd.DataFrame({
    "artifact": ["best_csv", "selected_jsonl", "all_starts_jsonl", "splits_json", "splits_csv"],
    "path": [best_csv_path, selected_jsonl_path, starts_jsonl_path, splits_json_path, splits_csv_path],
})

,artifact,path
0,best_csv,/Users/jgv/PycharmProjects/cylindrical-data-lv...
1,selected_jsonl,/Users/jgv/PycharmProjects/cylindrical-data-lv...
2,all_starts_jsonl,/Users/jgv/PycharmProjects/cylindrical-data-lv...
3,splits_json,/Users/jgv/PycharmProjects/cylindrical-data-lv...
4,splits_csv,/Users/jgv/PycharmProjects/cylindrical-data-lv...


In [7]:
metric_summary = (
    best_df
    .groupby(["candidate_id", "display_name", "candidate_model_type", "candidate_family", "K", "L"], dropna=False)
    .agg(
        n_splits=("split_seed", "nunique"),
        successful_best_fits=("success", "sum"),
        train_bic_mean=("train_bic", "mean"),
        train_bic_std=("train_bic", "std"),
        heldout_log_likelihood_mean=("heldout_log_likelihood", "mean"),
        heldout_log_likelihood_std=("heldout_log_likelihood", "std"),
        heldout_avg_log_likelihood_mean=("heldout_avg_log_likelihood", "mean"),
        heldout_avg_log_likelihood_std=("heldout_avg_log_likelihood", "std"),
        heldout_bic_mean=("heldout_bic", "mean"),
        converged_best_fits=("converged", "sum"),
        median_n_iter=("n_iter", "median"),
        mean_fit_time=("fit_time", "mean"),
    )
    .reset_index()
    .sort_values(["heldout_avg_log_likelihood_mean", "train_bic_mean"], ascending=[False, True])
)

metric_summary

,candidate_id,display_name,candidate_model_type,candidate_family,candidate_k,candidate_l,n_splits,successful_best_fits,train_bic_mean,train_bic_std,heldout_log_likelihood_mean,heldout_log_likelihood_std,heldout_avg_log_likelihood_mean,heldout_avg_log_likelihood_std,heldout_bic_mean,converged_best_fits,median_n_iter,mean_fit_time
7,vmf_gauss_tl_6x4,"vMF/Gaussian two-layer, (K,L)=(6,4)",two_layer_mixture,vmf_gaussian,6,4.0,100,100,1.120936e+06,777.759654,-139933.126041,397.916934,-3.736632,0.010626,282509.466639,100,55.0,72.095529
3,gauss_vmf_tl_6x4,"Gaussian/vMF two-layer, (K,L)=(6,4)",two_layer_mixture,gaussian_vmf,6,4.0,100,100,1.130905e+06,507.773742,-141315.161339,440.616133,-3.773536,0.011766,283946.664590,100,48.0,55.219924
6,vmf_gauss_tl_3x2,"vMF/Gaussian two-layer, (K,L)=(3,2)",two_layer_mixture,vmf_gaussian,3,2.0,100,100,1.141672e+06,483.746390,-142745.828839,486.469189,-3.811739,0.012990,286176.155472,100,24.0,11.512179
1,full_cyl_k6,"Full cylindrical mixture, K=6",cylindrical_mixture,full,6,NaN,100,100,1.142931e+06,1961.330118,-142812.398766,485.376763,-3.813517,0.012961,286751.586207,100,60.5,24.801997
5,ind_cyl_k6,"Independent cylindrical mixture, K=6",cylindrical_mixture,independent,6,NaN,100,100,1.147177e+06,1222.091171,-143398.213970,448.103447,-3.829160,0.011966,287544.110146,100,57.0,23.586767
0,full_cyl_k3,"Full cylindrical mixture, K=3",cylindrical_mixture,full,3,NaN,100,100,1.166897e+06,518.099324,-145880.422216,409.916000,-3.895442,0.010946,292318.973402,100,26.0,8.616454
2,gauss_vmf_tl_3x2,"Gaussian/vMF two-layer, (K,L)=(3,2)",two_layer_mixture,gaussian_vmf,3,2.0,100,100,1.170154e+06,522.884475,-146286.086737,397.466804,-3.906275,0.010614,293035.525826,100,57.0,24.691906
4,ind_cyl_k3,"Independent cylindrical mixture, K=3",cylindrical_mixture,independent,3,NaN,100,100,1.184171e+06,509.054968,-148050.099306,425.186574,-3.953379,0.011354,296468.774347,100,34.0,9.167407


In [8]:
best_by_split = (
    best_df
    .sort_values(["split_seed", "heldout_avg_log_likelihood", "train_bic"], ascending=[True, False, True])
    .groupby("split_seed", as_index=False)
    .head(1)
    .loc[:, [
        "split_seed",
        "candidate_id",
        "display_name",
        "em_start",
        "em_seed",
        "train_bic",
        "heldout_log_likelihood",
        "heldout_avg_log_likelihood",
        "converged",
        "n_iter",
    ]]
)

best_by_split

,split_seed,candidate_id,display_name,em_start,em_seed,train_bic,heldout_log_likelihood,heldout_avg_log_likelihood,converged,n_iter
7,0,vmf_gauss_tl_6x4,"vMF/Gaussian two-layer, (K,L)=(6,4)",0,178652633,1.120640e+06,-139999.909626,-3.738415,True,49
15,1,vmf_gauss_tl_6x4,"vMF/Gaussian two-layer, (K,L)=(6,4)",0,1633819610,1.120133e+06,-140353.269698,-3.747851,True,52
23,2,vmf_gauss_tl_6x4,"vMF/Gaussian two-layer, (K,L)=(6,4)",0,955738732,1.121047e+06,-139699.069039,-3.730382,True,63
31,3,vmf_gauss_tl_6x4,"vMF/Gaussian two-layer, (K,L)=(6,4)",0,1855101298,1.121037e+06,-139551.554554,-3.726443,True,58
39,4,vmf_gauss_tl_6x4,"vMF/Gaussian two-layer, (K,L)=(6,4)",0,1469786722,1.120010e+06,-139686.059282,-3.730034,True,81
...,...,...,...,...,...,...,...,...,...,...
767,95,vmf_gauss_tl_6x4,"vMF/Gaussian two-layer, (K,L)=(6,4)",0,336560733,1.121930e+06,-139602.437150,-3.727801,True,51
775,96,vmf_gauss_tl_6x4,"vMF/Gaussian two-layer, (K,L)=(6,4)",0,725761806,1.122369e+06,-139653.054861,-3.729153,True,55
783,97,vmf_gauss_tl_6x4,"vMF/Gaussian two-layer, (K,L)=(6,4)",0,1506069679,1.121922e+06,-139580.521793,-3.727216,True,54
791,98,vmf_gauss_tl_6x4,"vMF/Gaussian two-layer, (K,L)=(6,4)",0,84387278,1.122042e+06,-139564.950834,-3.726800,True,56
